In [1]:
import os
import glob
import time
import json
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict, Optional
from pageindex import PageIndexClient

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

GROQ_MODEL = "openai/gpt-oss-120b"

llm = ChatGroq(api_key=GROQ_API_KEY, model=GROQ_MODEL, temperature=0)
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

d:\Research Assistant Intelligent System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_DIR = "./pdfs"
pdf_paths = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
print(f"Found {len(pdf_paths)} PDFs:")
for p in pdf_paths:
    print(f"  {p}")

doc_records = {}  # doc_id -> {"path": ..., "filename": ..., "status": ...}

for path in pdf_paths:
    print(f"Uploading: {path}")
    result = pi_client.submit_document(path)
    doc_id = result["doc_id"]
    doc_records[doc_id] = {
        "path": path,
        "filename": os.path.basename(path),
        "status": "submitted"
    }
    print(f"  -> Document ID: {doc_id}")

Found 4 PDFs:
  ./pdfs\An Exploration of ECAPA-TDNN.pdf
  ./pdfs\DS-TTS.pdf
  ./pdfs\Improvement Speaker Similarity for.pdf
  ./pdfs\VOCALIS_preprint.pdf
Uploading: ./pdfs\An Exploration of ECAPA-TDNN.pdf
  -> Document ID: pi-cmrzeksbs007y01pgk5anlkv4
Uploading: ./pdfs\DS-TTS.pdf
  -> Document ID: pi-cmrzektfa007z01pgodj7bsaq
Uploading: ./pdfs\Improvement Speaker Similarity for.pdf
  -> Document ID: pi-cmrzekuqq008001pg1ru5b9jo
Uploading: ./pdfs\VOCALIS_preprint.pdf
  -> Document ID: pi-cmrzekws6008101pgbi9uw4t2


In [3]:
pending = set(doc_records.keys())

while pending:
    for doc_id in list(pending):
        status_result = pi_client.get_document(doc_id)
        status = status_result.get("status")
        doc_records[doc_id]["status"] = status

        filename = doc_records[doc_id]["filename"]
        print(f"[{filename}] Status: {status}")

        if status == "completed":
            print(f"  -> Tree Index ready for {filename}")
            pending.remove(doc_id)
        elif status == "failed":
            print(f"  -> Processing FAILED for {filename}")
            pending.remove(doc_id)

    if pending:
        time.sleep(5)

completed_docs = {k: v for k, v in doc_records.items() if v["status"] == "completed"}
failed_docs = {k: v for k, v in doc_records.items() if v["status"] == "failed"}

print(f"\nCompleted: {len(completed_docs)} | Failed: {len(failed_docs)}")

[An Exploration of ECAPA-TDNN.pdf] Status: processing
[Improvement Speaker Similarity for.pdf] Status: processing
[VOCALIS_preprint.pdf] Status: processing
[DS-TTS.pdf] Status: processing
[An Exploration of ECAPA-TDNN.pdf] Status: processing
[Improvement Speaker Similarity for.pdf] Status: processing
[VOCALIS_preprint.pdf] Status: processing
[DS-TTS.pdf] Status: processing
[An Exploration of ECAPA-TDNN.pdf] Status: completed
  -> Tree Index ready for An Exploration of ECAPA-TDNN.pdf
[Improvement Speaker Similarity for.pdf] Status: completed
  -> Tree Index ready for Improvement Speaker Similarity for.pdf
[VOCALIS_preprint.pdf] Status: processing
[DS-TTS.pdf] Status: completed
  -> Tree Index ready for DS-TTS.pdf
[VOCALIS_preprint.pdf] Status: processing
[VOCALIS_preprint.pdf] Status: completed
  -> Tree Index ready for VOCALIS_preprint.pdf

Completed: 4 | Failed: 0


In [4]:
document_trees = {}

for doc_id, info in completed_docs.items():
    filename = info["filename"]

    print(f"\n{'=' * 60}")
    print(f"Fetching tree for: {filename}")
    print(f"Document ID: {doc_id}")

    tree_result = pi_client.get_tree(doc_id, node_summary=True)
    pageindex_tree = tree_result.get("result", [])

    # Save the tree for later use
    document_trees[doc_id] = {
        "filename": filename,
        "tree": pageindex_tree
    }

    print(f"Top level sections: {len(pageindex_tree)}")

    print("\nRaw Tree:")
    if pageindex_tree:
        print(json.dumps(pageindex_tree[0], indent=2))
    else:
        print("No tree returned.")


Fetching tree for: An Exploration of ECAPA-TDNN.pdf
Document ID: pi-cmrzeksbs007y01pgk5anlkv4
Top level sections: 1

Raw Tree:
{
  "title": "An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This study evaluates the performance of three speaker encoders\u2014H/ASP, x-vector, and ECAPA-TDNN\u2014within a YourTTS-based zero-shot multi-speaker TTS system using Czech speech data. Through subjective listening tests and objective cosine distance measurements, the researchers found that the original H/ASP encoder consistently outperformed the others, indicating that popular speaker recognition embeddings like ECAPA-TDNN do not necessarily enhance speaker similarity in TTS and highlighting the necessity of empirical validation when repurposing such models.",
  "text": "# An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS\n\nMarie Kune\u016

In [5]:
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview"""
    for node in nodes:
        prefix = " " * indent + ("|_ " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")

        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


# Print the tree for every completed document
for doc_id, data in document_trees.items():
    print("\n" + "=" * 70)
    print(f"Document: {data['filename']}")
    print(f"Document ID: {doc_id}")
    print("=" * 70)

    print_tree(data["tree"])


Document: An Exploration of ECAPA-TDNN.pdf
Document ID: pi-cmrzeksbs007y01pgk5anlkv4
[0000] An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS (p.1)
 |_ [0001] 1 Introduction (p.1)
 |_ [0002] 2 TTS system (p.3)
  |_ [0003] 2.1 Training Data (p.3)
  |_ [0004] 2.2 Target speaker data (p.4)
  |_ [0005] 2.3 Speaker encoders (p.4)
 |_ [0006] 3 Listening test (p.5)
 |_ [0007] 4 Objective evaluation (p.7)
 |_ [0008] 5 Conclusions (p.10)
 |_ [0009] References (p.10)

Document: DS-TTS.pdf
Document ID: pi-cmrzektfa007z01pgodj7bsaq
[0000] DS-TTS: Zero-Shot Speaker Style Adaptation from Voice Clips via Dynamic Dual-Style Feature Modulation (p.1)
[0001] I. INTRODUCTION (p.1)
[0002] II. RELATED WORK (p.2)
[0003] III. DS-TTS (p.3)
 |_ [0004] A. Problem Formulation (p.3)
 |_ [0005] B. Model Architecture Overview (p.3)
 |_ [0006] C. Dual-Style Encoding Network (p.3)
 |_ [0007] D. Dynamic Generator Network (p.4)
[0008] IV. EXPERIMENT SETUP (p.6)
[0009] V. R

In [6]:
def compress_tree(nodes: list) -> list:
    """Compress a single doc's tree: title + truncated summary, keep nesting.
    doc_id/filename removed here — already known from the parent grouping."""
    out = []
    for n in nodes:
        entry = {
            "id": n["node_id"],
            "title": n["title"],
            "page": n.get("page_index", "?"),
            "sum": (n.get("summary") or n.get("prefix_summary") or "")[:200],  # cap length
        }
        if n.get("nodes"):
            entry["children"] = compress_tree(n["nodes"])
        out.append(entry)
    return out

In [7]:
def llm_tree_search(query: str, document_trees: dict) -> dict:
    compressed_all = []
    for doc_id, data in document_trees.items():
        compressed_all.append({
            "doc_id": doc_id,
            "filename": data["filename"],
            "tree": compress_tree(data["tree"])  # no longer needs doc_id/filename passed in
        })

    # Compact JSON — no indent, minimal separators
    tree_json = json.dumps(compressed_all, separators=(",", ":"))

    prompt = f"""You are given a query and the tree structures of one or more research papers.
Identify which node_ids (across ANY of the documents) most likely contain the answer.
If the query requires comparing multiple papers, select relevant nodes from each relevant paper.
Think step by step about which sections are relevant.

Query: {query}

Documents:
{tree_json}

Reply only in this format:
{{
    "thinking": "<your step by step reasoning>",
    "node_list": [
        {{"doc_id": "...", "node_id": "..."}}
    ]
}}"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return json.loads(response.content)

In [8]:
def flatten_tree(nodes: List[Dict], doc_id: str, filename: str, parent_title: str = None) -> List[Dict]:
    """
    Recursively flattens PageIndex's nested tree into a flat list of
    citation-ready section records.
    """
    flat_nodes = []

    for node in nodes:
        summary = node.get("summary") or node.get("prefix_summary") or ""

        title = node["title"]
        full_title = f"{parent_title} > {title}" if parent_title else title

        flat_nodes.append({
            "doc_id": doc_id,
            "filename": filename,
            "node_id": node["node_id"],
            "section_title": full_title,
            "summary": summary,
            "page_index": node.get("page_index"),
            "text": node.get("text", ""),
        })

        children = node.get("nodes", [])
        if children:
            flat_nodes.extend(
                flatten_tree(children, doc_id, filename, parent_title=full_title)
            )

    return flat_nodes


document_index = []

for doc_id, data in document_trees.items():
    filename = data["filename"]
    flat_nodes = flatten_tree(data["tree"], doc_id, filename)
    document_index.extend(flat_nodes)

print(f"Total indexed sections across all docs: {len(document_index)}")
for node in document_index[:5]:
    print(f"[{node['doc_id']}] {node['section_title']} (p.{node['page_index']}) — {node['summary'][:80]}...")

Total indexed sections across all docs: 64
[pi-cmrzeksbs007y01pgk5anlkv4] An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS (p.1) — This study evaluates the performance of three speaker encoders—H/ASP, x-vector, ...
[pi-cmrzeksbs007y01pgk5anlkv4] An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS > 1 Introduction (p.1) — This text introduces zero-shot multi-speaker TTS and the critical role of speake...
[pi-cmrzeksbs007y01pgk5anlkv4] An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS > 2 TTS system (p.3) — This section details the use of the YourTTS system, a VITS-based model utilizing...
[pi-cmrzeksbs007y01pgk5anlkv4] An Exploration of ECAPA-TDNN and x-vector Speaker Representations in Zero-shot Multi-speaker TTS > 2 TTS system > 2.1 Training Data (p.3) — The text details the 'SPT-MGW' Czech speech dataset used for TTS model training,...
[pi-cmrzek

In [9]:
def fetch_full_text(selected_nodes: list, flat_index: list) -> list:
    """
    Given the LLM's selected {doc_id, node_id} pairs, pull the full text
    from the flat document_index (built by flatten_tree earlier).
    """
    lookup = {(n["doc_id"], n["node_id"]): n for n in flat_index}
    results = []
    for sel in selected_nodes:
        key = (sel["doc_id"], sel["node_id"])
        if key in lookup:
            results.append(lookup[key])
    return results

In [14]:
def is_answerable(search_result: dict) -> bool:
    """Simple check: did the navigation step find any relevant nodes?"""
    return len(search_result.get("node_list", [])) > 0

In [15]:
def generate_answer(query: str, full_sections: list, chat_history: list = None) -> dict:
    """
    Takes the fetched full-text sections and produces a cited answer.
    Returns dict with 'answer' and 'citations'.
    """
    chat_history = chat_history or []

    context = "\n\n".join(
        f"[Source: {s['filename']}, Section: {s['section_title']}]\n{s['text']}"
        for s in full_sections
    )

    history_text = "\n".join(
        f"{m['role']}: {m['content']}" for m in chat_history
    )

    prompt = f"""Answer the user's question using ONLY the context provided below. Do not use outside knowledge.
Every claim you make must be followed by a citation in this exact format: (filename, Section: section_title).
If different sources disagree or provide nuance, reflect that clearly rather than oversimplifying.

Conversation so far:
{history_text}

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke([HumanMessage(content=prompt)])

    citations = [
        f"{s['filename']} — {s['section_title']}" for s in full_sections
    ]

    return {
        "answer": response.content,
        "citations": citations,
    }

In [ ]:
def fallback_response(query: str) -> dict:
    return {
        "answer": "No related documents found for this query.",
        "citations": [],
        "needs_arxiv_fallback": True,
    }

In [17]:
def answer_query(query: str, document_trees: dict, document_index: list, chat_history: list = None) -> dict:
    """
    Full pipeline: navigate -> check sufficiency -> fetch -> generate (or fallback).
    This is the Streamlit-ready function.
    """
    search_result = llm_tree_search(query, document_trees)

    if not is_answerable(search_result):
        result = fallback_response(query)
        result["thinking"] = search_result.get("thinking", "")
        return result

    full_sections = fetch_full_text(search_result["node_list"], document_index)
    result = generate_answer(query, full_sections, chat_history)
    result["needs_arxiv_fallback"] = False
    result["thinking"] = search_result.get("thinking", "")
    return result

In [1]:
chat_history = []